# Unified Bombcell Runner (Grant)

Single entrypoint replacing batch, single-probe, and NP2.0 rerun notebooks.

#### General pip install for loading .env

In [ ]:
#!pip install python-dotenv

#### Imports

In [26]:
from pathlib import Path
from pathlib import Path
import importlib
import numpy as np
import pandas as pd
from pathlib import Path
import subprocess

import nwb_data_prep as prep
# import pca_plotting as plots
# import verify_data as verify
# Load mouse_name from .env
import os
from dotenv import load_dotenv

# Always reload local modules so notebook uses latest patched code.
prep = importlib.reload(prep)
# plots = importlib.reload(plots)

print('pca_data_prep path:', Path(prep.__file__).resolve())
if not hasattr(prep, 'extract_probe_letters'):
    raise AttributeError(
        'Loaded pca_data_prep does not expose extract_probe_letters. '
        'Restart kernel and re-run this cell, then confirm it points to master/pca_data_prep.py.'
    )


pca_data_prep path: C:\Users\user\Documents\github\bombcell\mice\Reach15\run_bc\nwb_data_prep.py


### Load data from .env

In [10]:
session_data_dic = prep.load_env()
session_data_dic

MOUSE loaded: Reach15




-- Behavioral Files --
BEHAVIORAL_FOLDER loaded: grant_reach15_swingDoor-christie
-- First Neuropixels File --
NP_FILE loaded: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
NWB_FILE loaded: NA
DATE loaded: 20260129
SESSION loaded: session003
BOMBCELL loaded: NA


-- Second Neuropixels File --
NP_FILE loaded: Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32
NWB_FILE loaded: NA
DATE_01 loaded: 20260129
SESSION_01 loaded: session004
BOMBCELL_01 loaded: NA


-- Third Neuropixels File --
NP_FILE_02 loaded: Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00
NWB_FILE_02 loaded: Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00
DATE_02 loaded: 20260201
SESSION_02 loaded: session007
BOMBCELL_02 loaded: bombcell_batch_20260304_1536


{'MOUSE': 'Reach15',
 'BEHAVIORAL_FOLDER': 'grant_reach15_swingDoor-christie',
 'NP_FILE': 'Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01',
 'DATE': '20260129',
 'SESSION': 'session003',
 'BOMBCELL': 'NA',
 'NWB_FILE': 'NA',
 'NP_FILE_01': 'Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32',
 'DATE_01': '20260129',
 'SESSION_01': 'session004',
 'BOMBCELL_01': 'NA',
 'NWB_FILE_01': 'NA',
 'NP_FILE_02': 'Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00',
 'DATE_02': '20260201',
 'SESSION_02': 'session007',
 'BOMBCELL_02': 'bombcell_batch_20260304_1536',
 'NWB_FILE_02': 'Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00'}

#### =========================================================
## STEP 1: Select Neuropixel session to run bombcell on
#### =========================================================


In [11]:
SESSION_TO_ANALYZE = 1

MOUSE, BEHAVIORAL_FOLDER, NP_FILE, NWB_FILE, DATE, SESSION, BOMBCELL = prep.session_to_analyze(
                                                                                        session_data_dic['MOUSE'], session_data_dic['BEHAVIORAL_FOLDER'], 
                                                                                        session_data_dic['NP_FILE'],session_data_dic['NWB_FILE'] ,session_data_dic['DATE'], session_data_dic['SESSION'],session_data_dic['BOMBCELL'],
                                                                                        session_data_dic['NP_FILE_01'], session_data_dic['NWB_FILE_01'], session_data_dic['DATE_01'], session_data_dic['SESSION_01'],session_data_dic['BOMBCELL_01'] ,
                                                                                        session_data_dic['NP_FILE_02'], session_data_dic['NWB_FILE_02'], session_data_dic['DATE_02'], session_data_dic['SESSION_02'],session_data_dic['BOMBCELL_02'],
                                                                                        session_selection=SESSION_TO_ANALYZE
                                                                                    )
                                                               


SESSION SELECTION:

NP_FILE: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
NWB_FILE: NA
DATE: 20260129
SESSION: session003
BEHAVIORAL_FOLDER: grant_reach15_swingDoor-christie
BOMBCELL: NA


### Set and verify bombcell paths using loaded session data from .env

In [27]:
def set_bc_paths(session_data_dic, MOUSE, BEHAVIORAL_FOLDER, DATE, SESSION, SESSION_TO_ANALYZE):
    # SET #1: Path to the NWB file for this session (on the neural data computer)
    NWB_PATH = Path(fr"H:\NWB_OUT\{NWB_FILE}")

    # SET #2: Path to the bombcell root folder for this session (on the neural data computer)
    BOMBCELL_ROOT_FOR_AUTO_BUILD = Path(fr"H:\Grant\Neuropixels\Kilosort_Recordings\{NP_FILE}\bombcell\{BOMBCELL}")

    # SET #3: Session name for labeling plots
    SESSION_NAME = NP_FILE

    # SET #4: Paths to trial index files (behavior acquisition computer)
    baseline_trials_index_path = rf"G:\Grant\behavior_data\DLC_net\{BEHAVIORAL_FOLDER}\videos\{DATE}\christielab\{SESSION}\{DATE}_christielab_{SESSION}_baseline_trial_numbers_tone2_aligned.npy"
    washout_trials_index_path = rf"G:\Grant\behavior_data\DLC_net\{BEHAVIORAL_FOLDER}\videos\{DATE}\christielab\{SESSION}\{DATE}_christielab_{SESSION}_washout_trial_numbers_tone2_aligned.npy"
    optoicalStim_trials_index_path = rf"G:\Grant\behavior_data\DLC_net\{BEHAVIORAL_FOLDER}\videos\{DATE}\christielab\{SESSION}\{DATE}_christielab_{SESSION}_stim_allowed_trial_numbers_tone2_aligned.npy"

    # SET #5: Auto-generate a session-specific config file from .env + selected session
    CONFIG_FILE, _ = prep.build_session_grant_config(
        session_data_dic=session_data_dic,
        session_selection=SESSION_TO_ANALYZE,
        verbose=True,
    )

    for required_path, label in [
        (baseline_trials_index_path, "baseline trials index"),
        (washout_trials_index_path, "washout trials index"),
        (optoicalStim_trials_index_path, "optical stim trials index"),
        (NWB_PATH, "NWB file"),
        (BOMBCELL_ROOT_FOR_AUTO_BUILD, "Bombcell root folder"),
        (CONFIG_FILE, "session config file"),
    ]:
        if not Path(required_path).exists():
            raise FileNotFoundError(f"Missing {label}: {required_path}")

    print("All required files/folders found.")
    print(f"NWB file: {NWB_PATH}")
    print(f"Bombcell root folder: {BOMBCELL_ROOT_FOR_AUTO_BUILD}")
    print(f"Session config file: {CONFIG_FILE}")

    return BOMBCELL_ROOT_FOR_AUTO_BUILD, NWB_PATH, CONFIG_FILE, SESSION_NAME

set_bc_paths(session_data_dic, MOUSE, BEHAVIORAL_FOLDER, DATE, SESSION, SESSION_TO_ANALYZE)


✅ All behavior trial index files found
Baseline trials index file: G:\Grant\behavior_data\DLC_net\grant_reach15_swingDoor-christie\videos\20260129\christielab\session003\20260129_christielab_session003_baseline_trial_numbers_tone2_aligned.npy
Washout trials index file: G:\Grant\behavior_data\DLC_net\grant_reach15_swingDoor-christie\videos\20260129\christielab\session003\20260129_christielab_session003_washout_trial_numbers_tone2_aligned.npy
Optoical stim trials index file: G:\Grant\behavior_data\DLC_net\grant_reach15_swingDoor-christie\videos\20260129\christielab\session003\20260129_christielab_session003_stim_allowed_trial_numbers_tone2_aligned.npy

✅ Config file found: c:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config.json

❌ NWB file not found at: H:\NWB_OUT\NA
❌ Bombcell root folder not found at: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\NA


(WindowsPath('H:/Grant/Neuropixels/Kilosort_Recordings/Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01/bombcell/NA'),
 WindowsPath('H:/NWB_OUT/NA'),
 WindowsPath('c:/Users/user/Documents/github/bombcell/mice/Reach15/configs/grant_recording_config.json'))

In [17]:
BOMBCELL_ROOT_FOR_AUTO_BUILD, NWB_PATH, CONFIG_FILE, SESSION_NAME = set_bc_paths(
    session_data_dic,
    MOUSE,
    BEHAVIORAL_FOLDER,
    DATE,
    SESSION,
    SESSION_TO_ANALYZE,
)


✅ All behavior trial index files found
Baseline trials index file: G:\Grant\behavior_data\DLC_net\grant_reach15_swingDoor-christie\videos\20260129\christielab\session003\20260129_christielab_session003_baseline_trial_numbers_tone2_aligned.npy
Washout trials index file: G:\Grant\behavior_data\DLC_net\grant_reach15_swingDoor-christie\videos\20260129\christielab\session003\20260129_christielab_session003_washout_trial_numbers_tone2_aligned.npy
Optoical stim trials index file: G:\Grant\behavior_data\DLC_net\grant_reach15_swingDoor-christie\videos\20260129\christielab\session003\20260129_christielab_session003_stim_allowed_trial_numbers_tone2_aligned.npy

✅ Config file found: c:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config.json

❌ NWB file not found at: H:\NWB_OUT\NA
❌ Bombcell root folder not found at: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\NA


#### =======================================
## RUN BOMB CELL ANALYSIS
#### =======================================



In [ ]:
# Select run mode and target probe (if applicable)
RUN_MODE = 'batch'  # batch | single_probe | np20_rerun
TARGET_PROBE = 'A'  # only used for single_probe
OVERWRITE = True

runner = Path('run_bombcell_unified.py')
cmd = ['python', str(runner), '--config', str(CONFIG_FILE), '--mode', RUN_MODE]
if RUN_MODE == 'single_probe':
    cmd += ['--target-probe', TARGET_PROBE]
if OVERWRITE:
    cmd.append('--overwrite')

print('Running:', ' '.join(cmd))
# subprocess.run(cmd, check=True)
cmd

Running: python run_bombcell_unified.py --config c:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config.json --mode batch --overwrite


['python',
 'run_bombcell_unified.py',
 '--config',
 'c:\\Users\\user\\Documents\\github\\bombcell\\mice\\Reach15\\configs\\grant_recording_config.json',
 '--mode',
 'batch',
 '--overwrite']

In [4]:
# new code
p = subprocess.run(cmd, text=True, capture_output=True)
print("Return code:", p.returncode)
print("\n--- STDOUT ---\n", p.stdout)
print("\n--- STDERR ---\n", p.stderr)
p.check_returncode()  # will raise after printing, if you still want it to


Return code: 0

--- STDOUT ---
 ✅ ipywidgets available - interactive GUI ready
Creating run root for mode 'batch' on 20260304 at 1536...

=== Probe A (NP2.0 Simplex Lobule & Interposed Nucleus (SIM/IP)) ===
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A
raw_file: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\continuous\Neuropix-PXI-100.ProbeA\continuous.dat
meta_file: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\Record Node 103\experiment1\recording1\structure.oebin
save_path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A\bombcell
Applying overrides for probe A: {'maxRPVviolations': 0.3

## View Results of BC run

In [5]:
bombcell_run = r'H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536'

In [8]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import bombcell as bc

analysis_dir = Path.cwd().resolve()
sys.path.insert(0, str(analysis_dir))
from post_analysis_setup import load_post_analysis_context

ctx = load_post_analysis_context(CONFIG_FILE)

staging_root = bombcell_run
print('staging_root:', staging_root)
probe_letters = list(ctx['probeLetters'])

for TARGET_PROBE in probe_letters:
    ks_dir = Path(staging_root) / f'kilosort4_{TARGET_PROBE}'
    save_path = ks_dir / 'bombcell'

    print('ks_dir:', ks_dir)
    print('save_path:', save_path)

    param, quality_metrics, _ = bc.load_bc_results(str(save_path))
    unit_type, unit_type_string = bc.qm.get_quality_unit_type(param, quality_metrics)
    qm_df = pd.DataFrame(quality_metrics).copy()
    qm_df['bombcell_label'] = unit_type_string
    qm_df['unit_index'] = np.arange(len(qm_df))

    if 'cluster_id' not in qm_df.columns:
        qm_df['cluster_id'] = qm_df['unit_index']

    print('Loaded units:', len(qm_df))
    print(qm_df['bombcell_label'].value_counts(dropna=False))

staging_root: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A
save_path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_A\bombcell
Loaded units: 554
bombcell_label
NOISE       392
MUA         101
NON-SOMA     51
GOOD         10
Name: count, dtype: int64
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_B
save_path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell\bombcell_batch_20260304_1536\kilosort4_B\bombcell
Loaded units: 